# Limpieza del dataset Google Play Store**Unidad II — Minería de Datos**Objetivo: dejar el dataset apto para un modelo, atacando cuatro problemas: duplicados de scraping,columnas numéricas guardadas como texto, símbolos de moneda y outliers de precio.

In [21]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"
df = pd.read_csv(url)

---## Pregunta A — Duplicados por scrapingEl scraper recorrió varias categorías y guardó la misma app en cada pasada. Como `App` es la llavelógica del registro, un nombre repetido implica una fila duplicada.

In [22]:
df.duplicated(subset=['App']).sum()

1181

In [23]:
antes = len(df)
df.drop_duplicates(subset=['App'], keep='first', inplace=True)

print(f"Registros antes:    {antes}")
print(f"Registros después:  {len(df)}")
print(f"Duplicados purgados: {antes - len(df)}")

Registros antes:    10841
Registros después:  9660
Duplicados purgados: 1181


### Respuesta A Se purgaron **1,181 registros duplicados**. El dataset pasó de **10,841** a **9,660** filas.

---## Pregunta B — `Installs` como textoLa columna `Installs` viene como texto (`"1,000,000+"`). Hay que quitar el `+` y la `,`,y convertir a numérico. Se usa `errors='coerce'` en vez de `.astype(int)` directo porquehay una fila corrupta (índice 10472) con columnas desalineadas que rompería la conversión.

In [24]:
df['Installs'] = pd.to_numeric(
    df['Installs'].str.replace('+', '', regex=False).str.replace(',', '', regex=False),
    errors='coerce'
)

print("Media Installs:", df['Installs'].mean())
print("Mediana Installs:", df['Installs'].median())

Media Installs: 7777506.732270421
Mediana Installs: 100000.0


### Respuesta B El promedio real es de **≈7.78 millones de installs**, muy por encima de la mediana (100,000).Esto indica un sesgo fuerte a la derecha: unas cuantas apps mega-populares jalan el promedio hacia arriba.

---## Pregunta C — `Price` con símbolo de dólarMisma lógica que en B: quitar el `$` y convertir a `float`.

In [25]:
df['Price'] = pd.to_numeric(
    df['Price'].astype(str).str.replace('$', '', regex=False),
    errors='coerce'
)

df['Price'].max()

400.0

 ### Pregunta D — Outliers de precio`Price.max()` da ~$400. Al filtrar `Price > 200` aparecen apps basura/broma (ej. la familia "I Am Rich").Si se dejan, un modelo podría interpretar que vender apps a $400 es un modelo de negocio normal.Por eso se conservan solo las apps con `Price < 50` y se exporta el resultado final.

In [26]:
caras = df[df['Price'] > 200]
print("Apps con Price > 200:", len(caras))
caras[['App', 'Price']]

Apps con Price > 200: 17


,App,Price
4197,most expensive app (H),399.99
4362,💎 I'm rich,399.99
4367,I'm Rich - Trump Edition,400.00
5351,I am rich,399.99
5354,I am Rich Plus,399.99
5355,I am rich VIP,299.99
5356,I Am Rich Premium,399.99
5357,I am extremely Rich,379.99
5358,I am Rich!,399.99
5359,I am rich(premium),399.99


In [27]:
antes_d = len(df)
df = df[df['Price'] < 50]

print(f"Registros antes del filtro: {antes_d}")
print(f"Registros después del filtro: {len(df)}")
print(f"Eliminados: {antes_d - len(df)}")

Registros antes del filtro: 9660
Registros después del filtro: 9636
Eliminados: 24


In [28]:
df.to_csv('playstore_limpio.csv', index=False)
print(df.shape)

(9636, 13)


### Respuesta D Se eliminaron 24 filas con precios absurdos.

El dataset final quedó en **9,636 filas**